In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===========================================
# 1. DATA PREPARATION AND GEOGRAPHICAL SIMULATION
# ===========================================
print("Data is being loaded and preparations are being made...")
df_raw = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
base_features = ['AT_solar_generation_actual', 'AT_wind_onshore_generation_actual', 'AT_load_actual_entsoe_transparency']
data_region_A = df_raw[base_features].copy().dropna()

def add_time_features(df):
    time_features = pd.DataFrame(index=df.index)
    time_features['hour'] = time_features.index.hour
    time_features['month'] = time_features.index.month
    time_features['hour_sin'] = np.sin(2 * np.pi * time_features['hour'] / 24.0)
    time_features['hour_cos'] = np.cos(2 * np.pi * time_features['hour'] / 24.0)
    time_features['month_sin'] = np.sin(2 * np.pi * time_features['month'] / 12.0)
    time_features['month_cos'] = np.cos(2 * np.pi * time_features['month'] / 12.0)
    return pd.concat([df, time_features[['hour_sin', 'hour_cos', 'month_sin', 'month_cos']]], axis=1)

data_region_A = add_time_features(data_region_A)

# Geographic Difference (Target Region) Simulation
data_region_B = data_region_A.copy()
data_region_B['AT_wind_onshore_generation_actual'] *= 0.70  
data_region_B['AT_load_actual_entsoe_transparency'] = data_region_B['AT_load_actual_entsoe_transparency'] * 0.85 + np.random.normal(0, 150, len(data_region_B))

scaler_A = MinMaxScaler(feature_range=(-1, 1))
data_scaled_A = scaler_A.fit_transform(data_region_A.values)

scaler_B = MinMaxScaler(feature_range=(-1, 1))
data_scaled_B = scaler_B.fit_transform(data_region_B.values)

def create_daily_sequences(data_array, lookback=48, horizon=24):
    X, y = [], []
    for i in range(len(data_array) - lookback - horizon + 1):
        X.append(data_array[i : (i + lookback), :])
        y.append(data_array[(i + lookback) : (i + lookback + horizon), :].flatten()) 
    return np.array(X), np.array(y)

X_A, y_A = create_daily_sequences(data_scaled_A, 48, 24)
X_B, y_B = create_daily_sequences(data_scaled_B, 48, 24)

# ===========================================
# 2. SPECIAL LOSS FUNCTION (SMOOTHNESS PENALTY)
# ===========================================
class SpatialSmoothnessLoss(nn.Module):
    def __init__(self, lambda_smooth=0.05):
        super(SpatialSmoothnessLoss, self).__init__()
        self.mse = nn.MSELoss()
        self.lambda_smooth = lambda_smooth # Fiziksel esneklik ceza katsayısı

    def forward(self, predictions, targets):
        base_loss = self.mse(predictions, targets)
        
        preds_reshaped = predictions.view(-1, 24, 7)
        diffs = preds_reshaped[:, 1:, :] - preds_reshaped[:, :-1, :]
        smoothness_penalty = torch.mean(diffs ** 2)
        
        return base_loss + (self.lambda_smooth * smoothness_penalty)

# ===========================================
# 3. STANDARD ARCHITECTURES
# ===========================================
INPUT_SIZE = 7
OUTPUT_SIZE = 7 * 24 

class Model_GRU(nn.Module):
    def __init__(self):
        super(Model_GRU, self).__init__()
        self.gru = nn.GRU(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, OUTPUT_SIZE)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

class Model_LSTM(nn.Module):
    def __init__(self):
        super(Model_LSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, OUTPUT_SIZE)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class Model_BiLSTM(nn.Module):
    def __init__(self):
        super(Model_BiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.fc = nn.Linear(128, OUTPUT_SIZE) 
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# ===========================================
# 4. CROSS VALIDATION (CV) + SPATIAL TESTING CYCLE
# ===========================================
def run_spatial_cv(ModelClass, model_name, use_smoothness_loss):
    n_splits = 3
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    wape_scores = []
    mae_scores = []
    rmse_scores = []
    ramp_scores = []
    
    loss_type = "Fiziksel Pürüzsüzlük (Smoothness Penalty)" if use_smoothness_loss else "Standart Hata (MSE)"
    print(f"\n--- {model_name} | {loss_type} | ÇAPRAZ DOĞRULAMA ---")
    
    fold = 1
    for train_index, test_index in tscv.split(X_A):
        fold_train_size = int(len(train_index) * 0.85)
        real_train_idx = train_index[:fold_train_size]
        val_idx = train_index[fold_train_size:]
        
        X_train_A = torch.from_numpy(X_A[real_train_idx]).float()
        y_train_A = torch.from_numpy(y_A[real_train_idx]).float()
        X_val_A = torch.from_numpy(X_A[val_idx]).float()
        y_val_A = torch.from_numpy(y_A[val_idx]).float()

        # TEST: REGION B (Rural / Diverse Geography)
        X_test_B = torch.from_numpy(X_B[test_index]).float()
        y_test_B = torch.from_numpy(y_B[test_index]).float()

        train_loader_A = DataLoader(TensorDataset(X_train_A, y_train_A), batch_size=64, shuffle=False)
        val_loader_A = DataLoader(TensorDataset(X_val_A, y_val_A), batch_size=64, shuffle=False)
        
        model = ModelClass().to(device)
        
        if use_smoothness_loss:
            criterion = SpatialSmoothnessLoss(lambda_smooth=0.05)
        else:
            criterion = nn.MSELoss()
            
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        
        best_val_loss = float('inf')
        best_weights = copy.deepcopy(model.state_dict())
        patience = 5
        epochs_no_improve = 0
        
        for epoch in range(25):
            model.train()
            for X_batch, y_batch in train_loader_A:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch)
                loss.backward()
                optimizer.step()
                
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_batch, y_batch in val_loader_A:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    val_loss += criterion(model(X_batch), y_batch).item()
            val_loss /= len(val_loader_A)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_weights = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve == patience:
                    break
                    
        # TESTING PHASE
        model.load_state_dict(best_weights)
        model.eval()
        with torch.no_grad():
            preds = model(X_test_B.to(device)).cpu().numpy()
            
        preds_mw = scaler_B.inverse_transform(preds.reshape(-1, INPUT_SIZE)).reshape(preds.shape)
        y_test_B_mw = scaler_B.inverse_transform(y_test_B.numpy().reshape(-1, INPUT_SIZE)).reshape(y_test_B.shape)

        actual_solar = y_test_B_mw[:, 0::INPUT_SIZE]
        actual_wind  = y_test_B_mw[:, 1::INPUT_SIZE]
        actual_load  = y_test_B_mw[:, 2::INPUT_SIZE]
        
        pred_solar = preds_mw[:, 0::INPUT_SIZE]
        pred_wind  = preds_mw[:, 1::INPUT_SIZE]
        pred_load  = preds_mw[:, 2::INPUT_SIZE]

        # NET LOAD CALCULATION
        actual_net_load = actual_load - (actual_solar + actual_wind)
        pred_net_load = pred_load - (pred_solar + pred_wind)
        
        net_load_mae = mean_absolute_error(actual_net_load, pred_net_load)
        net_load_mse = mean_squared_error(actual_net_load, pred_net_load)
        net_load_rmse = np.sqrt(net_load_mse)
        net_load_wape = (net_load_mae / np.mean(np.abs(actual_net_load))) * 100

        # FLEXIBILITY (RAMPING) CALCULATION
        actual_ramp = np.abs(np.diff(actual_net_load, axis=1))
        pred_ramp = np.abs(np.diff(pred_net_load, axis=1))
        ramp_mae = mean_absolute_error(actual_ramp, pred_ramp)
        
        print(f"  [Fold {fold}] WAPE: %{net_load_wape:.2f} | MAE: {net_load_mae:.2f} MW | RMSE: {net_load_rmse:.2f} MW | Ramping: {ramp_mae:.2f} MW")
        
        wape_scores.append(net_load_wape)
        mae_scores.append(net_load_mae)
        rmse_scores.append(net_load_rmse)
        ramp_scores.append(ramp_mae)
        
        fold += 1

    print(f"[{model_name}] NİHAİ ORTALAMALAR:")
    print(f" -> WAPE: %{np.mean(wape_scores):.2f}")
    print(f" -> MAE:  {np.mean(mae_scores):.2f} MW")
    print(f" -> RMSE: {np.mean(rmse_scores):.2f} MW")
    print(f" -> Ramping MAE: {np.mean(ramp_scores):.2f} MW")
    print("="*65)

# ===========================================
# 5. START THE EXPERIMENT
# ===========================================
print("\n=============================================================================")
print("SPATIAL AWARENESS + PHYSICAL SMOOTH TEST BEGINS")
print("===============================================================================")

# GRU
run_spatial_cv(Model_GRU, "GRU", use_smoothness_loss=False)
run_spatial_cv(Model_GRU, "GRU", use_smoothness_loss=True)

# LSTM
run_spatial_cv(Model_LSTM, "LSTM", use_smoothness_loss=False)
run_spatial_cv(Model_LSTM, "LSTM", use_smoothness_loss=True)

# Bi-LSTM
run_spatial_cv(Model_BiLSTM, "Bi-LSTM", use_smoothness_loss=False)
run_spatial_cv(Model_BiLSTM, "Bi-LSTM", use_smoothness_loss=True)